In [1]:
from dofusapi import DofusAPI
from dataprocessor import DataProcessor
# from visualizer import EquipmentVisualizer
from utils import CacheManager


In [17]:
import numpy as np
def print_graph_info(graph):
    print("=== GRAPH ANALYSIS ===")
    print(f"Total equipments: {len(equipments)}")
    print(f"Total resources: {len(resource_usage)}")
    print(f"Total edges: {graph.number_of_edges()}")
    components = list(nx.connected_components(graph))
    print(f"Connected components: {len(components)}")
    print(f"Largest component size: {max(len(c) for c in components)}")
    # Analyze resource usage patterns
    usage_counts = [len(equipments) for equipments in resource_usage.values()]
    print(f"Avg resources per equipment: {graph.number_of_edges() / len(equipments):.2f}")
    print(f"Avg equipment per resource: {np.mean(usage_counts):.2f}")
    equipment_degrees = [deg for _, deg in graph.degree() if _ in [e.ankama_id for e in equipments]]
    print(f"Equipment degree - Min: {min(equipment_degrees)}, Max: {max(equipment_degrees)}")

In [2]:
equipments = DofusAPI.get_all_equipments()

✅ 151 équipements récupérés avec succès


In [ ]:
import networkx as nx
from collections import defaultdict

G = nx.Graph()
resource_usage = defaultdict(list)
for equipment in equipments:
    for resource in equipment.recipe:
        G.add_edge(equipment.ankama_id, resource.resource_id)
        resource_usage[resource.resource_id].append(equipment.ankama_id)

In [20]:
bipartite = nx.Graph()
for equip in equipments:            
    # Add equipment node
    bipartite.add_node(equip.ankama_id, bipartite=0, type="equipment")
    
    # Add resource nodes and edges
    for resource in equip.recipe:
        if not bipartite.has_node(resource.resource_id):
            bipartite.add_node(resource.resource_id, bipartite=1, type="resource")
        bipartite.add_edge(equip.ankama_id, resource.resource_id)

print(f"Bipartite graph has {bipartite.number_of_nodes()} nodes and {bipartite.number_of_edges()} edges.")

Bipartite graph has 457 nodes and 690 edges.


In [19]:
import numpy as np
import networkx as nx
from collections import defaultdict

def bipartite_modularity(G, partition):
    """
    Calcule la modularité bipartite selon la définition de Barber
    """
    # Séparer les nœuds par type
    equipment_nodes = [n for n in G.nodes() if G.nodes[n].get('bipartite') == 0]
    resource_nodes = [n for n in G.nodes() if G.nodes[n].get('bipartite') == 1]
    
    m = G.number_of_edges()
    Q = 0.0
    
    for u, v in G.edges():
        if u in equipment_nodes and v in resource_nodes:
            # Vérifier si les deux nœuds sont dans le même groupe
            if partition[u] == partition[v]:
                k_u = G.degree(u)
                k_v = G.degree(v)
                Q += 1 - (k_u * k_v) / (2 * m)
        elif v in equipment_nodes and u in resource_nodes:
            if partition[v] == partition[u]:
                k_v = G.degree(v)
                k_u = G.degree(u)
                Q += 1 - (k_v * k_u) / (2 * m)
    
    return Q / (2 * m)

def bilouvain_community_detection(G, resolution=1.0, random_state=None):
    """
    Algorithme BiLouvain adapté pour graphes bipartis
    """
    equipment_nodes = [n for n in G.nodes() if G.nodes[n].get('bipartite') == 0]
    resource_nodes = [n for n in G.nodes() if G.nodes[n].get('bipartite') == 1]
    
    # Phase 1: Projection pondérée sur les équipements
    equipment_projection = nx.bipartite.weighted_projected_graph(G, equipment_nodes)
    
    # Utiliser Louvain standard sur la projection comme point de départ
    import community as community_louvain
    partition = community_louvain.best_partition(equipment_projection, 
                                               resolution=resolution, 
                                               random_state=random_state)
    
    # Étendre la partition aux ressources (affecter chaque ressource au groupe majoritaire de ses équipements connectés)
    resource_partition = {}
    for resource in resource_nodes:
        connected_equipments = list(G.neighbors(resource))
        if connected_equipments:
            # Trouver le groupe le plus fréquent parmi les équipements connectés
            groups = [partition[eq] for eq in connected_equipments]
            most_common_group = max(set(groups), key=groups.count)
            resource_partition[resource] = most_common_group
        else:
            resource_partition[resource] = -1  # Ressource isolée
    
    # Fusionner les partitions
    full_partition = {**partition, **resource_partition}
    
    return full_partition


In [22]:
partition = bilouvain_community_detection(bipartite)

In [23]:
bipartite_modularity(bipartite, partition)

0.41384320520898804